# N149 · 迭代器、惰性展开与查看下一项

**目标：** 维护迭代状态而不提前物化全部输出。

**先修：** N009, N053, N069, N014。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 迭代协议；显式栈；缓存一项；hasNext幂等；摊还；嵌套结构。

## 从问题到算法

迭代器不必预先生成全部输出。BST保留尚未访问的左链；peek只缓存一个已取出但尚未交付的值；嵌套列表用迭代器栈保存当前展开路径。hasNext允许整理内部状态，但重复调用不能跳过下一个值。用唯一哨兵区分“没有缓存”与真正的None值。

## 最小实现

**本章接口：** `BSTIterator`、`PeekingIterator`、`NestedIterator`

In [1]:
class TreeNode:
    def __init__(self,val=0,left=None,right=None):
        self.val,self.left,self.right=val,left,right

def tree_from_level(values):
    """Construct a finite binary tree from level order with None placeholders."""
    from collections import deque
    if not values or values[0] is None: return None
    root=TreeNode(values[0]); queue=deque([root]); i=1
    while queue and i<len(values):
        node=queue.popleft()
        if values[i] is not None: node.left=TreeNode(values[i]); queue.append(node.left)
        i+=1
        if i<len(values):
            if values[i] is not None: node.right=TreeNode(values[i]); queue.append(node.right)
            i+=1
    return root
class BSTIterator:
    def __init__(self,root): self.stack=[]; self._push_left(root)
    def _push_left(self,node):
        while node: self.stack.append(node); node=node.left
    def hasNext(self): return bool(self.stack)
    def next(self):
        if not self.stack: raise StopIteration
        node=self.stack.pop(); self._push_left(node.right); return node.val
    def __iter__(self): return self
    __next__=next

class PeekingIterator:
    def __init__(self,iterable): self.iterator=iter(iterable); self.empty=object(); self.cached=self.empty
    def _fill(self):
        if self.cached is self.empty:
            try: self.cached=next(self.iterator)
            except StopIteration: return False
        return True
    def hasNext(self): return self._fill()
    def peek(self):
        if not self._fill(): raise StopIteration
        return self.cached
    def next(self):
        value=self.peek(); self.cached=self.empty; return value
    def __iter__(self): return self
    __next__=next

class NestedIterator:
    def __init__(self,nested_list):
        if not isinstance(nested_list,list): raise TypeError('nested Python list required')
        self.stack=[iter(nested_list)]; self.empty=object(); self.cached=self.empty
    def _fill(self):
        if self.cached is not self.empty: return True
        while self.stack:
            try: value=next(self.stack[-1])
            except StopIteration: self.stack.pop(); continue
            if isinstance(value,int): self.cached=value; return True
            if not isinstance(value,list): raise TypeError('only integers and lists are supported')
            self.stack.append(iter(value))
        return False
    def hasNext(self): return self._fill()
    def next(self):
        if not self._fill(): raise StopIteration
        value=self.cached; self.cached=self.empty; return value
    def __iter__(self): return self
    __next__=next

## 正确性、前提与复杂度

BST栈顶始终是下一中序节点，弹出后只需处理其右子树左链。peek缓存交付前的唯一值，不改变可见输出顺序。嵌套迭代器把递归调用栈显式保存，每个列表元素只取出一次；空列表被跳过但整数不会被hasNext重复消耗。

**代价：** BST构建O(h)，完整遍历O(n)，每次next均摊O(1)，空间O(h)。嵌套结构总耗时按包括空列表在内的结构节点数O(N)，辅助O(深度)。本章NestedIterator使用原生Python嵌套列表；平台NestedInteger需替换取整数/子列表访问器，未声称该签名可原样提交。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

it=PeekingIterator([1,None,3]); rows=[]
while it.hasNext(): rows.append((it.peek(),it.peek(),it.next()))
show_table(['第一次peek','第二次peek','真正next'],rows)
print('嵌套输出',list(NestedIterator([[],[1,[2,[]]],3])))

第一次peek,第二次peek,真正next
1,1,1
None,None,None
3,3,3


嵌套输出 [1, 2, 3]


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert list(BSTIterator(tree_from_level([7,3,15,None,None,9,20])))==[3,7,9,15,20]
assert list(BSTIterator(None))==[]
it=PeekingIterator([None,0,False]); assert it.peek() is None and it.peek() is None and it.next() is None
assert list(it)==[0,False] and not it.hasNext()
for data,expected in [([],[]),([[],[[]]],[]),([1,[4,[6]]],[1,4,6]),([[],[1],[],[2,[3]]],[1,2,3])]:
    it=NestedIterator(data); out=[]
    while it.hasNext(): assert it.hasNext(); out.append(it.next())
    assert out==expected
    try: it.next()
    except StopIteration: pass
    else: raise AssertionError('exhaustion must raise StopIteration')
nested=9
for _ in range(1500): nested=[nested]
assert list(NestedIterator(nested))==[9]
root=TreeNode(0); node=root
for value in range(1,1500): node.right=TreeNode(value); node=node.right
assert list(BSTIterator(root))==list(range(1500))
print('N149: 所有本章断言通过')

N149: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 反复调用hasNext不消耗元素。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 说明next最坏与均摊成本不同。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 173. Binary Search Tree Iterator（canonical）
- 284. Peeking Iterator（canonical）
- 341. Flatten Nested List Iterator（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。